# Training code 

## Import

In [1]:
import aegnn
import wandb
import datetime
import lightning.pytorch as pl
import torch

from pathlib import Path
from lightning.pytorch.loggers import WandbLogger

import os

In [5]:
os.environ["AEGNN_DATA_DIR"] = "/home/kevin-imagine/Documents/event_graph/data"
print(os.environ["AEGNN_DATA_DIR"])

/home/kevin-imagine/Documents/event_graph/data


## Classification

### Parameters

In [6]:
params = {"model":"graph_res",
          "task":"",
          "dim":3,
          "seed":12345,
          "lr": 1e-3,
          "weight_decay":5e-3,
          "eta_min":0.0,
          "max_epochs":100,
          "Trainer":{
            "max_epochs":1,
            "overfit_batches":0,
            "log_every_n_steps":1,
            "gradient_clip_val":0,
            "limit_train_batches":None,
            "limit_val_batches":None
          },
          "log-gradients":True,
          "profile":True,
          "debug":True,
          "gpu": True if torch.cuda.is_available() else False,
          "dataset":"ncars",
          "Data":{
              "batch_size":32,
              "num_workers":8,
              "pin_memory":False
          }          
}

### Model and dataset

In [7]:
data_module = aegnn.datasets.NCars(shuffle = True,
                                   **params["Data"],
                                   )

In [8]:
data_module.batch_size

32

In [9]:
data_module.setup()

In [10]:
data_module.train_dataset[10]

Data(x=[2095, 1], pos=[2095, 3], file_id='sequence_384', label=[1], y=[1], edge_index=[2, 54418], edge_attr=[54418, 3])

In [11]:
model = aegnn.models.RecognitionModel(params["model"],
                                      params["dataset"],
                                      num_classes=data_module.num_classes,
                                      img_shape=data_module.dims,
                                      dim = params["dim"],
                                      bias = True,
                                      root_weight = True,                                     
                                      )


### Loggers

In [12]:
log_dir = "../training_log_aegnn/"

In [13]:
project = f"aegnn-{params['dataset']}-{params['task']}"
experiment_name = datetime.datetime.now().strftime("%Y%m%d%H%M%S")


In [21]:
wandb_logger = WandbLogger(project="aegnn",
                        #    groupe = "ncars",
                           name=experiment_name)

In [22]:
loggers = []
loggers.append(wandb_logger)
# loggers = pl.loggers.LoggerCollection(loggers)

In [16]:
checkpoint_path = os.path.join(log_dir, "checkpoints", params["dataset"], params["task"], experiment_name)
Path(checkpoint_path).mkdir(parents=True,exist_ok=True)

In [17]:
callbacks = [
        pl.callbacks.LearningRateMonitor(),
        aegnn.utils.callbacks.BBoxLogger(classes=data_module.classes),
        # aegnn.utils.callbacks.PHyperLogger(args),
        aegnn.utils.callbacks.EpochLogger(),
        aegnn.utils.callbacks.FileLogger([model, model.model, data_module]),
        aegnn.utils.callbacks.FullModelCheckpoint(dirpath=checkpoint_path)
    ]

In [23]:
trainer_kwargs = {
    "accelerator":"gpu" if params["gpu"] else "cpu",
    "profiler": "simple" if params["profile"] else False,
}

In [24]:
trainer = pl.Trainer(logger=loggers,
                     callbacks=callbacks,
                     **params["Trainer"],
                     **trainer_kwargs
                     )

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [25]:
trainer.fit(model, datamodule=data_module)

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/training/*
RAW FILer length 12961


100%|██████████| 12961/12961 [00:00<00:00, 385193.61it/s]

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/validation/*
RAW FILer length 2462



100%|██████████| 2462/2462 [00:00<00:00, 260464.52it/s]
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/kevin-imagine/.netrc.
wandb: Currently logged in as: kevin-hoarau (kevin-hoarau-cole-normale-sup-rieure-paris-saclay) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | criterion | CrossEntropyLoss | 0      | train | 0    
1 | model     | GraphRes         | 30.4 K | train | 0    
---------------------------------------------------------------
30.4 K    Trainable params
0         Non-trainable params
30.4 K    Total params
0.121     Total estimated model params size (MB)
40        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.
FIT Profiler Report

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Action                                                                                                                                                             	|  Mean duration (s)	|  Num calls      	|  Total time (s) 	|  Percentage %   	|
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Total                                                                                                                                               

In [2]:
import torch

In [3]:
torch.cuda.is_available()

True

In [4]:
torch.cuda.get_device_name(0)

'NVIDIA RTX PRO 1000 Blackwell Generation Laptop GPU'

In [ ]:
import lightning.pytorch as pl